In [2]:
# notebook : semdetect_smoke.ipynb — cellule 1 : environnement et classifieur
import importlib.metadata as md
import json
import logging

import numpy as np
import pandas as pd
import torch

from src.config import PROJ_ROOT
from src.data.dataset_semdetect import SemDetectDataset
from src.detector.sem_detect.semdetect.claim_extractor import CLAIM_CATEGORIES, ClaimExtraction
from src.detector.sem_detect.semdetect.classifier import CLASS_NAMES, SemDetectClassifier
from src.detector.sem_detect.semdetect.embedding_generator import EmbeddingGenerator
from src.detector.sem_detect.semdetect.feature_builder import build_feature_vector, get_semantic_embeddings
from src.detector.sem_detect.semdetect.textual_features import TextualFeatureExtractor

logging.getLogger("httpx").setLevel(logging.WARNING)

# chaîne explicite : device="cuda" seul déclenche le mode multi-GPU (processus spawn) du code Sem-Detect
DEVICE = "cuda:0"
MODEL_PKL = PROJ_ROOT / "src/detector/sem_detect/models/lightgbm_best_model.pkl"

for pkg in ("torch", "transformers", "sentence-transformers", "scikit-learn", "lightgbm", "shap"):
    print(f"{pkg:<22} {md.version(pkg)}")

clf = SemDetectClassifier(str(MODEL_PKL))
print(f"\ntau = {clf.similarity_threshold}")
print(f"classes_ LightGBM = {getattr(clf.model, 'classes_', None)} | CLASS_NAMES = {CLASS_NAMES}")
print(f"features = {clf._feature_names}")
print(f"scaler n_features_in_ = {clf.scaler.n_features_in_}")

INFO:src.detector.sem_detect.semdetect.classifier:Loading Sem-Detect model from /home/cyaiche/generated_text_detection_hub/src/detector/sem_detect/models/lightgbm_best_model.pkl


torch                  2.11.0+cu126
transformers           5.8.0
sentence-transformers  5.5.0
scikit-learn           1.7.2
lightgbm               4.7.0
shap                   0.49.1


/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
INFO:src.detector.sem_detect.semdetect.classifier:SHAP TreeExplainer initialized
INFO:src.detector.sem_detect.semdetect.classifier:Model loa


tau = 0.8
classes_ LightGBM = [0 1 2] | CLASS_NAMES = ['ai', 'rewrite', 'human']
features = ['Column_0', 'Column_1', 'Column_2', 'Column_3', 'Column_4', 'Column_5', 'Column_6', 'Column_7', 'Column_8']
scaler n_features_in_ = 9


In [3]:
# notebook : semdetect_smoke.ipynb — cellule 1b : contrôle numérique du scaler désérialisé
rng = np.random.default_rng(0)
x = rng.normal(size=(5, clf.scaler.n_features_in_))
manual = (x - clf.scaler.mean_) / clf.scaler.scale_
assert np.allclose(clf.scaler.transform(x), manual), "StandardScaler désérialisé incohérent"
print("scaler OK | mean_ =", np.round(clf.scaler.mean_, 4))
print("           scale_ =", np.round(clf.scaler.scale_, 4))

scaler OK | mean_ = [ 0.1138  0.5403  0.6726  0.5368  2.7749  7.7362 -5.3514  0.9793  1.6155]
           scale_ = [0.1261 0.4021 0.0537 0.0381 0.4478 2.2938 1.9738 0.0097 0.2549]


In [4]:
# notebook : semdetect_smoke.ipynb — cellule 2 : trois cibles et leurs références
te = SemDetectDataset(config={
    "name": "sem_detect", "repo_id": "Sem-Detect/ML_Conferences-Peer-Reviews",
    "revision": "18b74b551670bff6b04e57570232a1694997863b",
    "split": "test", "conference": None, "date": None, "generator": None,
    "label_mode": "three_class", "balance": "none", "subset_size": None,
    "val_fraction": None, "val_seed": 0,
    "no_duplicate": True, "keep_meta": True, "with_paper": False, "keep_claims": True,
    "exclude_ids_path": None, "seed": 0,
}).to_pandas()


def parse_claims(raw: str) -> ClaimExtraction:
    d = json.loads(raw) if raw else {}
    unknown = set(d) - set(CLAIM_CATEGORIES)
    if unknown:
        raise KeyError(f"catégories de claims inattendues : {sorted(unknown)} ; attendu {CLAIM_CATEGORIES}")
    return ClaimExtraction(**{c: list(d.get(c) or []) for c in CLAIM_CATEGORIES})


def references(target: pd.Series, k: int = 3) -> pd.DataFrame:
    """Reviews IA du même papier, même note, autre modèle que la cible (protocole §3.2)."""
    pool = te[(te["class"] == "ai")
              & (te["submission_id"] == target["submission_id"])
              & (te["rating"] == target["rating"])
              & (te["generator"] != target["generator"])]
    return pool.sort_values("generator").head(k)


te["n_semantic"] = te["claim_extraction"].map(lambda r: len(parse_claims(r).semantic_claims()))
examples = pd.concat([
    te[(te["class"] == c) & (te["n_semantic"] >= 4)].head(1) for c in ("human", "rewrite", "ai")
])
print(examples[["review-id", "class", "generator", "rating", "n_semantic"]].to_string(index=False))

INFO:src.data.dataset_semdetect:Loading Sem-Detect dataset from Sem-Detect/ML_Conferences-Peer-Reviews (revision=18b74b551670bff6b04e57570232a1694997863b)
split  class    reason          
train  human    collision           22
       rewrite  rewrite_orphelin    88
dtype: int64
INFO:src.data.dataset_semdetect:Déduplication : 0 doublons supprimés
INFO:src.data.dataset_semdetect:Sem-Detect | split=['test'] conf=['ICLR', 'NeurIPS'] years=['2021', '2022'] label_mode=three_class balance=none | 4410/22165 reviews
INFO:src.data.dataset_semdetect:Répartition class x label :
label       0     1     2
label_3c                 
ai          0     0  1364
human     606     0     0
rewrite     0  2440     0


                                                     review-id   class        generator     rating  n_semantic
                                        ICLR-2021-yHeg4PbFHh-2   human            human Borderline          13
     ICLR-2022-vwLLQ-HwqhZ-rewrite_0_gemini-2.5-flash_2fa675ae rewrite gemini-2.5-flash Borderline           9
NeurIPS-2022-uP9RiC4uVcR-ai_gemini-2.5-pro_Borderline_66cad95f      ai   gemini-2.5-pro Borderline          22


In [5]:
# notebook : semdetect_smoke.ipynb — cellule 3 : features et prédiction
embedder = EmbeddingGenerator(device=DEVICE)
textual = TextualFeatureExtractor(device=DEVICE)  # télécharge Mistral-7B-Instruct-v0.3 au premier appel


def semantic_embeddings(claims: ClaimExtraction, min_embeddings: int):
    return get_semantic_embeddings(embedder.embed_claims(claims.to_dict()), min_embeddings=min_embeddings)


rows = []
for _, t in examples.iterrows():
    refs = references(t)
    target_sem = semantic_embeddings(parse_claims(t["claim_extraction"]), 4)
    ref_sems = [s for s in (semantic_embeddings(parse_claims(r), 1) for r in refs["claim_extraction"]) if s]
    stats = textual.compute(t["content"])
    feats = build_feature_vector(target_sem, ref_sems, stats, clf.similarity_threshold)
    pred = clf.predict(feats)
    rows.append({
        "vrai": t["class"], "prédit": pred.predicted_class,
        **{f"p_{name}": round(p, 3) for name, p in pred.probabilities.items()},
        "refs": list(refs["generator"]),
        "features": np.round(feats, 3).tolist(),
    })

print(pd.DataFrame(rows).to_string(index=False))
print(f"\nVRAM max : {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f} Go")

INFO:src.detector.sem_detect.semdetect.embedding_generator:Embedding device: cuda:0
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from Qwen/Qwen3-Embedding-0.6B.
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4960.79it/s]
INFO:sentence_transformers.base.model:Loaded 1 prompt with these keys: ['query']
INFO:src.detector.sem_detect.semdetect.embedding_generator:Embedding model loaded
INFO:src.detector.sem_detect.semdetect.textual_features:Textual features device: cuda:0
INFO:src.detector.sem_detect.semdetect.textual_features:Loading textual features model: mistralai/Mistral-7B-Instruct-v0.3
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:06<00:00, 42.09it/s] 
INFO:src.detector.sem_detect.semdetect.textual_features:Textual features model loaded


   vrai  prédit  p_ai  p_rewrite  p_human                                                                 refs                                                         features
  human   human 0.000      0.013    0.987                 [deepseek.v3-v1:0, gemini-2.5-flash, gemini-2.5-pro]    [0.0, 0.0, 0.559, 0.589, 2.639, 19.391, -3.395, 0.939, 2.547]
rewrite rewrite 0.005      0.979    0.016   [deepseek.v3-v1:0, gemini-2.5-pro, qwen.qwen3-235b-a22b-2507-v1:0] [0.111, 0.826, 0.678, 0.493, 2.303, 8.227, -5.906, 0.976, 1.561]
     ai      ai 0.993      0.007    0.000 [deepseek.v3-v1:0, gemini-2.5-flash, qwen.qwen3-235b-a22b-2507-v1:0] [0.318, 0.835, 0.74, 0.491, 3.135, 5.988, -11.016, 0.985, 1.246]

VRAM max : 16.5 Go


In [6]:
# notebook : semdetect_aprdb.ipynb — cellule 1 : faisabilité de R2 et formatage par classe
import logging
import re

import numpy as np
import pandas as pd

from src.data.dataset_aprdb import AprdbDataset

logging.getLogger("httpx").setLevel(logging.WARNING)

APRDB_BASE = {
    "name": "aprdb",
    "description": "AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt",
    "conference": None, "date": None, "keep_meta": True,
    "split": "test", "with_pdf": False, "seed": 0,
}
pool = AprdbDataset(config={**APRDB_BASE, "subset_size": None}).to_pandas()     # réservoir de références
evald = AprdbDataset(config={**APRDB_BASE, "subset_size": 10000}).to_pandas()   # cibles évaluées par SAGE
print(f"réservoir : {len(pool)} reviews | cibles SAGE : {len(evald)}")
print(pool.groupby(["label", "generator"]).size().unstack(0).fillna(0).astype(int), "\n")

# ---- couverture des références par review-id ----
mgt = pool[pool["label"] == 1]
gens = mgt.groupby("review-id")["generator"].agg(lambda s: set(s))
n_gens = gens.map(len)
print("générateurs MGT distincts par review-id :")
print(n_gens.value_counts().sort_index().to_string(), "\n")


def n_refs_available(row) -> int:
    g = gens.get(row["review-id"], set())
    return len(g) if row["label"] == 0 else len(g - {row["generator"]})


evald["n_refs"] = evald.apply(n_refs_available, axis=1)
print("cibles SAGE avec >= 3 références disponibles :")
print(evald.assign(ok=evald["n_refs"] >= 3).groupby("label")["ok"].agg(["sum", "count", "mean"]), "\n")

# ---- formatage par classe (ce que strip_markdown laisse ou retire) ----
patterns = {
    "titres #": r"(?m)^\s*#+\s",
    "gras **": r"\*\*",
    "puces - * •": r"(?m)^\s*[-*•]\s+",
    "listes numérotées": r"(?m)^\s*\d+[.)]\s+",
    "préambule LLM": r"(?i)^\s*(here is|here's|sure|certainly)\b",
}
fmt = pd.DataFrame({k: pool["content"].str.contains(p, regex=True) for k, p in patterns.items()})
fmt["label"] = pool["label"].to_numpy()
print("part des reviews contenant chaque marqueur :")
print(fmt.groupby("label").mean().round(3).T)
print("\nlongueur (caractères) :", pool.groupby("label")["content"].str.len().describe()[["mean", "50%"]].round(0).to_dict()
      if False else pool.assign(n=pool["content"].str.len()).groupby("label")["n"].median().to_dict())

/data2/cyaiche/cache_dir
Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'split': 'test', 'with_pdf': False, 'seed': 0, 'subset_size': None}


Map: 100%|██████████| 3884/3884 [00:00<00:00, 9967.76 examples/s] 


Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'split': 'test', 'with_pdf': False, 'seed': 0, 'subset_size': 10000}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_13af569ff61302c3c2d7fbb1eea34954.pt
réservoir : 101890 reviews | cibles SAGE : 9468
label          0      1
generator              
claude     17011  17012
gemini         2  16986
gpt4o          0  17014
llama          0  16944
qwen           0  16921 

générateurs MGT distincts par review-id :
generator
3        3
4      187
5    16823 

cibles SAGE avec >= 3 références disponibles :
        sum  count    mean
label                     
0      4468   4468  1.0000
1      4999   5000  0.9998 



/tmp/ipykernel_3572639/2832118599.py:48: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  fmt = pd.DataFrame({k: pool["content"].str.contains(p, regex=True) for k, p in patterns.items()})


part des reviews contenant chaque marqueur :
label                  0      1
titres #           0.036  0.125
gras **            0.041  0.374
puces - * •        0.224  0.696
listes numérotées  0.158  0.280
préambule LLM      0.000  0.000

longueur (caractères) : {0: 1986.0, 1: 2595.0}
